# Day 34 — **Python ↔ Java Interop (Py4J & REST)**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

Barclays runs on the JVM. Your Python agents will need answers from Java systems — a pricing engine, a risk library, a core-banking calc. Two ways in: **Py4J** (in-process bridge to a running JVM) and **REST** (call a Java service over HTTP). Knowing *when* to reach for each is the point.

Py4J needs a live JVM gateway we can't boot in a notebook, so below is a **faithful mock of the Py4J call surface** (same `gateway.jvm...` shape you'd write for real) plus the REST alternative wired to a local handler — no network, fully runnable.

Today:
1. The **Py4J call surface** — what talking to a JVM object looks like.
2. **REST interop** — the option you'll actually pick 90% of the time, and why.
3. A **thin adapter** so your agent tool doesn't care which transport it is.

## 1. The Py4J call surface

Py4J starts a `JavaGateway` that talks to a `GatewayServer` running inside the JVM. From Python you then use Java objects *as if they were Python* — `gateway.jvm.<class>`. This mock preserves that surface so the shape is real.

In [1]:
class _JVM:
    """Mimics Py4J's gateway.jvm — attribute access maps to Java classes."""
    class com:
        class barclays:
            class risk:
                class LoanCalculator:
                    def __init__(self): pass
                    def monthlyRepayment(self, principal, annualRate, months):
                        r = annualRate / 12
                        return principal * r * (1 + r) ** months / ((1 + r) ** months - 1)

class JavaGateway:
    """Stand-in for py4j.java_gateway.JavaGateway()."""
    def __init__(self): self.jvm = _JVM()

# --- exactly how the real Py4J code reads ---
gateway = JavaGateway()                                   # real: JavaGateway()
calc = gateway.jvm.com.barclays.risk.LoanCalculator()     # instantiate a Java object
repayment = calc.monthlyRepayment(40_000, 0.079, 36)      # call a Java method
print(f"Java LoanCalculator -> monthly GBP {repayment:.2f}")

Java LoanCalculator -> monthly GBP 1251.61


☝ The seductive part of Py4J is that `calc.monthlyRepayment(...)` looks like a plain Python call — but every call is a **socket round-trip to the JVM**, and the JVM must be alive in the same box with the gateway server started. That tight coupling (shared host, matching lifecycle, JVM classpath must contain your class) is why Py4J shines for *co-located, high-frequency, low-latency* calls — think a Spark driver — and is a poor fit for calling a service on another machine.

## 2. REST interop — the default choice

For a Java service that lives elsewhere (its own deployment, its own team), you call it over HTTP. It's loosely coupled, language-agnostic, independently deployable, and firewall-friendly. Here the "Java endpoint" is a local handler so it runs offline; in reality it's an `httpx` call (Day 35).

In [2]:
import json

def java_service_endpoint(body: dict) -> dict:
    """Pretend Spring Boot controller: POST /loan/repayment -> JSON."""
    p, rate, months = body["principal"], body["annualRate"], body["months"]
    r = rate / 12
    m = p * r * (1 + r) ** months / ((1 + r) ** months - 1)
    return {"monthlyRepayment": round(m, 2), "currency": "GBP"}

def call_java_rest(principal: float, annual_rate: float, months: int) -> dict:
    payload = {"principal": principal, "annualRate": annual_rate, "months": months}
    # real: resp = httpx.post("https://risk.internal.bank/loan/repayment", json=payload, timeout=10)
    #       resp.raise_for_status(); return resp.json()
    return java_service_endpoint(payload)

print(call_java_rest(40_000, 0.079, 36))

{'monthlyRepayment': 1251.61, 'currency': 'GBP'}


☝ Same answer, radically different coupling. REST needs no shared host, no JVM in *your* process, no classpath — just a URL and a JSON contract, so the Java team can redeploy or rewrite in Kotlin and your agent never notices. The trade is per-call latency (network + serialisation) and you must handle timeouts/retries yourself (Days 35, 37). For agent tools calling bank services, **REST is the default**; reserve Py4J for the rare co-located, chatty, latency-critical case.

## 3. A transport-agnostic adapter

Your agent tool shouldn't hard-code *how* Java is reached. Hide it behind one interface; swap Py4J vs REST by config. This is the Strategy pattern (Day 14) applied to interop.

In [3]:
from typing import Protocol

class RepaymentCalculator(Protocol):
    def monthly(self, principal: float, rate: float, months: int) -> float: ...

class Py4JCalculator:
    def __init__(self, gateway): self._calc = gateway.jvm.com.barclays.risk.LoanCalculator()
    def monthly(self, principal, rate, months):
        return self._calc.monthlyRepayment(principal, rate, months)

class RestCalculator:
    def monthly(self, principal, rate, months):
        return call_java_rest(principal, rate, months)["monthlyRepayment"]

def agent_tool(calc: RepaymentCalculator, principal: float) -> str:
    m = calc.monthly(principal, 0.079, 36)
    verdict = "affordable" if m <= 2000 else "over cap"
    return f"GBP {m:.2f}/mo — {verdict}"

# swap transports without touching the tool logic
print("via Py4J:", agent_tool(Py4JCalculator(gateway), 40_000))
print("via REST:", agent_tool(RestCalculator(), 40_000))

via Py4J: GBP 1251.61/mo — affordable
via REST: GBP 1251.61/mo — affordable


☝ `agent_tool` depends on the `RepaymentCalculator` **Protocol**, not on Py4J or httpx — so the transport becomes a one-line config switch and, crucially, the tool is **testable without any JVM or network** (inject a fake `monthly`). That's the pattern to carry into agent design: isolate the integration behind an interface so the agent logic stays pure and unit-testable, and the messy interop lives in one swappable adapter.

## Recap

| Question | Answer |
|---|---|
| Call Java in-process, low-latency, co-located? | **Py4J** — `gateway.jvm.<class>` |
| Call a Java *service* elsewhere? | **REST** (httpx + JSON) — the default |
| Py4J cost | shared host + live JVM + classpath coupling |
| REST cost | per-call latency; you own timeout/retry |
| Keep agent testable | hide transport behind a `Protocol` adapter |

**Rule:** default to REST for service calls; reach for Py4J only when co-location and latency demand it. Tomorrow (Day 35): `httpx.AsyncClient` — the right way to make those REST calls from an async agent.